In [1]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import xgboost as xgb
from sklearn.neighbors import KNeighborsRegressor
#FOR AUGMENTED_NEW

import pandas as pd
import numpy as np

In [2]:

aug_factor = 3 #augmented has been done initially for aug_factor = 2, then for aug_factor = 3

doc_train_augmented = pd.read_csv (f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TRAIN_AUG{aug_factor}.csv")
doc_test = pd.read_csv(f"D:\Shukra_sir\MLAllTraining\ForTraining\TrainableWithBandRatios\AugmentedWithBR\DOC_AUG_WITH_BR\DOCWithBR_TEST_AUG{aug_factor}.csv")

doc_train_augmented.shape, doc_test.shape

((2732, 55), (171, 55))

In [3]:
doc_train_augmented ['result_DOC_log'] = np.log1p (doc_train_augmented ['result_DOC'])
doc_test ['result_DOC_log'] = np.log1p (doc_test ['result_DOC'])
doc_train_augmented

,Unnamed: 0,sample_date,B1,B11,B12,B2,B3,B4,B5,B6,...,B3_div_B2,ND_B4_B3,ND_B2_B3,ND_B6_B8,day_of_year,doy_sin,doy_cos,hour_sin,hour_cos,result_DOC_log
0,0,2024-05-08 14:00:00,0.037800,0.168600,0.105100,0.040100,0.063600,0.050700,0.094400,0.220200,...,1.585995,-0.112860,-0.226613,-0.130503,129,0.796183,-0.605056,-0.500000,-0.866025,0.741937
1,1,2019-08-20 15:00:00,0.068100,0.144400,0.130300,0.057300,0.075900,0.073100,0.098200,0.105800,...,1.324584,-0.018792,-0.139639,0.109596,232,-0.752667,-0.658402,-0.707107,-0.707107,1.308333
2,2,2022-02-01 20:55:00,0.015100,0.165100,0.131400,0.033200,0.053800,0.056300,0.095100,0.146200,...,1.620433,0.022706,-0.236779,-0.071156,32,0.523416,0.852078,-0.866025,0.500000,0.741937
3,3,2018-11-06 21:25:00,0.007100,0.026500,0.019000,0.019100,0.030400,0.022600,0.023400,0.014200,...,1.591540,-0.147167,-0.228278,-0.080904,310,-0.811539,0.584298,-0.707107,0.707107,1.481605
4,4,2018-11-06 19:20:00,0.001400,0.010400,0.010300,0.013300,0.020300,0.008900,0.006200,0.000000,...,1.526201,-0.390398,-0.208327,0.000000,310,-0.811539,0.584298,-0.965926,0.258819,1.193922
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2727,2727,2019-08-20 16:00:00,0.160048,0.347841,0.301870,0.194634,0.206942,0.225023,0.253643,0.254485,...,1.063231,0.041857,-0.030649,-0.043629,232,-0.752667,-0.658402,-0.866025,-0.500000,1.536867
2728,2728,2021-08-17 15:45:00,0.106466,0.173790,0.156926,0.132880,0.156900,0.169819,0.182550,0.194755,...,1.180757,0.039542,-0.082891,0.040882,229,-0.717677,-0.696376,-0.707107,-0.707107,1.547563
2729,2729,2023-05-24 17:43:00,0.041518,0.115512,0.094602,0.046671,0.063934,0.065133,0.090442,0.112315,...,1.369847,0.009291,-0.156073,-0.035528,144,0.615285,-0.788305,-0.965926,-0.258819,1.410987
2730,2730,2018-06-19 19:24:00,0.051536,0.046751,0.038356,0.069572,0.074987,0.053739,0.051703,0.041847,...,1.077817,-0.165069,-0.037458,-0.020389,170,0.213521,-0.976938,-0.965926,0.258819,1.029619


In [4]:

TARGET = "result_DOC_log" #FOR DOC
TARGET1 = "result_DOC"


In [5]:
best_params_A = {'n_neighbors': 3, 'weights': 'distance', 'p': 1}
best_params_B = {'n_neighbors': 3, 'weights': 'distance', 'p': 1}

In [6]:
# Features

X_with_meteo_columns = [
    'B2','B3','B4','B5','B6','B7','B8','B8A','B9',
    "B4_minus_B3",
    "B4_div_B3",
    "ND_B4_B3",
    'Temperature','DewPoint','v10n',
    'doy_sin','doy_cos',
    'latitude','longitude'
]

X_without_meteo_columns = [
    'B2','B3','B4','B5','B6','B7','B8','B8A','B9',
    "B4_minus_B3",
    "B4_div_B3",
    "ND_B4_B3",
    'doy_sin','doy_cos',
    'latitude','longitude'
]


In [7]:

from scipy.stats import ttest_rel
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler


In [8]:

#paired t-test: for scaled

# Data (LOG SPACE target used in training)

#A: model with meteo
#B: Model without meteo

X_train_A = doc_train_augmented[X_with_meteo_columns]
X_train_B = doc_train_augmented[X_without_meteo_columns]
y_train = doc_train_augmented[TARGET]   # log scale

X_test_A = doc_test[X_with_meteo_columns]
X_test_B = doc_test[X_without_meteo_columns]
y_test_log = doc_test[TARGET]

# real scale
y_test_real = np.expm1(y_test_log)

# CV setup
rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=42)

r2_log_A, r2_log_B = [], []
r2_real_A, r2_real_B = [], []

for train_idx, _ in rkf.split(X_train_A):

    # split raw (unscaled) data
    X_tr_A = X_train_A.iloc[train_idx]
    X_tr_B = X_train_B.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]

    
    # SCALE Model A features
    
    scaler_A = StandardScaler()
    X_tr_A_scaled = scaler_A.fit_transform(X_tr_A)
    X_test_A_scaled = scaler_A.transform(X_test_A)

    # SCALE Model B features
    scaler_B = StandardScaler()
    X_tr_B_scaled = scaler_B.fit_transform(X_tr_B)
    X_test_B_scaled = scaler_B.transform(X_test_B)

    
    # Model A (with meteo)
    model_A = KNeighborsRegressor(
        **best_params_A,
    )

    model_A.fit(X_tr_A_scaled, y_tr)

    pred_log_A = model_A.predict(X_test_A_scaled)
    pred_real_A = np.expm1(pred_log_A)

    r2_log_A.append(r2_score(y_test_log, pred_log_A))
    r2_real_A.append(r2_score(y_test_real, pred_real_A))

    
    # Model B (without meteo)
    model_B = KNeighborsRegressor(
        **best_params_B,
    )

    model_B.fit(X_tr_B_scaled, y_tr)

    pred_log_B = model_B.predict(X_test_B_scaled)
    pred_real_B = np.expm1(pred_log_B)

    r2_log_B.append(r2_score(y_test_log, pred_log_B))
    r2_real_B.append(r2_score(y_test_real, pred_real_B))

# Statistical tests
t_log, p_log = ttest_rel(r2_log_A, r2_log_B)
t_real, p_real = ttest_rel(r2_real_A, r2_real_B)

In [9]:
# Results for scaled
print(" LOG SCALE ")

print("p-value:", p_log)
print("t-value:", t_log)

print("\nREAL SCALE")
print("p-value:", p_real)
print("t-value:", t_real)


 LOG SCALE 
p-value: 3.17747364648387e-08
t-value: -6.560971452422771

REAL SCALE
p-value: 1.23815695341846e-18
t-value: -13.904804370218093


In [10]:
#for scaled
print(" LOG SCALE ")
print("Mean R2 A:", np.mean(r2_log_A))
print("Mean R2 B:", np.mean(r2_log_B))

print("\nREAL SCALE")
print("Mean R2 A:", np.mean(r2_real_A))
print("Mean R2 B:", np.mean(r2_real_B))

 LOG SCALE 
Mean R2 A: 0.3246317652506007
Mean R2 B: 0.36735164338114373

REAL SCALE
Mean R2 A: 0.18352790071518027
Mean R2 B: 0.285352569621511


In [11]:
from scipy.stats import wilcoxon

w_log, p_log = wilcoxon(r2_log_A, r2_log_B)
w_real, p_real = wilcoxon(r2_real_A, r2_real_B)

print("===== LOG SCALE =====")
print("Wilcoxon statistic:", w_log)
print("p-value:", p_log)

print("\n===== REAL SCALE =====")
print("Wilcoxon statistic:", w_real)
print("p-value:", p_real)

===== LOG SCALE =====
Wilcoxon statistic: 113.0
p-value: 2.6378053519238165e-08

===== REAL SCALE =====
Wilcoxon statistic: 0.0
p-value: 1.7763568394002505e-15


In [12]:
def corrected_paired_ttest(scores_A, scores_B, n_train, n_test):
    """
    Nadeau & Bengio (2003) corrected resampled paired t-test.
    Accounts for train/test overlap in repeated k-fold CV.
    """
    diff = np.array(scores_A) - np.array(scores_B)
    n = len(diff)
    mean_diff = np.mean(diff)
    var_diff = np.var(diff, ddof=1)  # sample variance

    correction = (1 / n) + (n_test / n_train)
    se_corrected = np.sqrt(correction * var_diff)

    t_stat = mean_diff / se_corrected
    # degrees of freedom = n - 1, same as standard paired t-test
    p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 1))

    return t_stat, p_value

from scipy import stats

# n_train = size of each fold's training set (~80% of doc_train_augmented, since 5-fold)
# n_test  = size of the fixed doc_test set
n_train = int(0.8 * len(X_train_A))   # training rows per fold
n_test  = len(X_test_A)               # your fixed pristine test set size

t_log_corr, p_log_corr = corrected_paired_ttest(r2_log_A, r2_log_B, n_train, n_test)
t_real_corr, p_real_corr = corrected_paired_ttest(r2_real_A, r2_real_B, n_train, n_test)

print("\n===== Corrected Paired t-test Results as per Nadeau and Bengio =====")
print(f"Log scale  — corrected t: {t_log_corr:.4f}, corrected p: {p_log_corr:.4g}")
print(f"Normal scale — corrected t: {t_real_corr:.4f}, corrected p: {p_real_corr:.4g}")


===== Corrected Paired t-test Results as per Nadeau and Bengio =====
Log scale  — corrected t: -2.9600, corrected p: 0.00473
Normal scale — corrected t: -6.2732, corrected p: 8.863e-08
